# Setup

In [1]:
import copy
import pandas as pd
import numpy as np
import time
import pickle
import warnings
import io
import contextlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, average_precision_score, brier_score_loss
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sktime.transformations.panel.catch22 import Catch22
from scipy.special import expit
from scipy import stats
from sklearn.impute import SimpleImputer
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [2]:
# Random state seed
RAND_STATE = 42

In [3]:
# Load the normalized data
output_dir = Path('../processed/')
output_dir = Path("/kaggle/input/datasets/brocklarson/normalized/") # TODO: change back after done with kaggle

data = np.load(output_dir / 'normalized_split.npz')
X_train_norm = data['X_train']
X_val_norm = data['X_val']
X_test_norm = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']

# with open(output_dir / 'norm_params.pkl', 'rb') as f_in:
#     norm_params = pickle.load(f_in)

print("Pre-processed data loaded successfully")

Pre-processed data loaded successfully


In [4]:
for name, X, y in [("TRAINING", X_train_norm, y_train),
                   ("VALIDATION", X_val_norm, y_val),
                   ("TEST", X_test_norm, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

TRAINING DATA
X shape:    (12455, 288, 10)
y shape:    (12455,)
NSEP count: 12337
SEP count:  118
Ratio:      104.55:1

VALIDATION DATA
X shape:    (1780, 288, 10)
y shape:    (1780,)
NSEP count: 1763
SEP count:  17
Ratio:      103.71:1

TEST DATA
X shape:    (3559, 288, 10)
y shape:    (3559,)
NSEP count: 3525
SEP count:  34
Ratio:      103.68:1



In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
torch.backends.cudnn.benchmark = True 

X_train_t = torch.as_tensor(X_train_norm, dtype=torch.float32, device=device)
X_val_t = torch.as_tensor(X_val_norm,   dtype=torch.float32, device=device)
X_test_t = torch.as_tensor(X_test_norm,  dtype=torch.float32, device=device)

device: cuda


### Catch22 Features

In [6]:
# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    return Catch22(catch24=False).fit_transform(X_chunk).to_numpy()

def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 200) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    # Safety net: replace any non-finite values with the channel's median
    if not np.isfinite(X).all():
        print(f"    ⚠️  non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]                       # view, so edits land in X
            bad = ~np.isfinite(col)
            col[bad] = np.median(col[~bad])

    # sktime expects (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(delayed(_catch22_chunk)(ch) for ch in chunks)
    X_feat = np.vstack(results)

    assert X_feat.shape == (n_samples, 22 * n_channels), f"Unexpected shape: {X_feat.shape}"
    return X_feat


# Extract features for each split──────────────────────────
print("Extracting Catch22 features …")
X_train_c22 = extract_catch22(X_train_norm, label="train")
X_val_c22   = extract_catch22(X_val_norm,   label="val")
X_test_c22  = extract_catch22(X_test_norm,  label="test")
print(f"train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


# ══════════════════════════════════════════════════════════════
# IMPUTE — fit on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

n_bad = [int((~np.isfinite(a)).sum()) for a in (X_train_c22, X_val_c22, X_test_c22)]

if any(n_bad):
    print(f"non-finite catch22 values — train: {n_bad[0]}  val: {n_bad[1]}  test: {n_bad[2]} → imputing")

    # Treat inf as missing too, so the imputer handles both
    X_train_c22, X_val_c22, X_test_c22 = [
        np.where(np.isfinite(a), a, np.nan) for a in (X_train_c22, X_val_c22, X_test_c22)
    ]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imp = SimpleImputer(strategy="median", keep_empty_features=True)
        X_train_c22 = imp.fit_transform(X_train_c22)
        X_val_c22   = imp.transform(X_val_c22)
        X_test_c22  = imp.transform(X_test_c22)
else:
    print("✓ No NaN or Inf in catch22 features — no imputation needed")

X_train_c22, X_val_c22, X_test_c22 = [
    a.astype(np.float32) for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"Ready: train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


CLIP_Z = 10
X_train_c22, X_val_c22, X_test_c22 = [
    np.clip(a, -CLIP_Z, CLIP_Z).astype(np.float32)
    for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"train col mean range: {X_train_c22.mean(0).min():.2g} to {X_train_c22.mean(0).max():.2g}")
print(f"train col std  range: {X_train_c22.std(0).min():.2g} to {X_train_c22.std(0).max():.2g}")
print(f"val   col max |z|   : {np.abs(X_val_c22).max():.1f}")

Extracting Catch22 features …
    train — 12455 samples in 63 chunks …
    val — 1780 samples in 9 chunks …
    test — 3559 samples in 18 chunks …
train (12455, 220)  val (1780, 220)  test (3559, 220)
non-finite catch22 values — train: 84320  val: 20276  test: 33898 → imputing
Ready: train (12455, 220)  val (1780, 220)  test (3559, 220)
train col mean range: -0.35 to 10
train col std  range: 0 to 3.8
val   col max |z|   : 10.0


In [7]:
# Catch22 inputs for the MLP: standardize with TRAIN statistics, then clip
mu = X_train_c22.mean(0)
sd = X_train_c22.std(0)
sd[sd < 1e-6] = 1.0                      # constant columns become all zeros instead of dividing by 0
print("constant catch22 columns:", int((X_train_c22.std(0) < 1e-6).sum()), "of", X_train_c22.shape[1])

prep = lambda A: torch.as_tensor(np.clip((A - mu) / sd, -CLIP_Z, CLIP_Z), dtype=torch.float32, device=device)
C22_train_t, C22_val_t, C22_test_t = prep(X_train_c22), prep(X_val_c22), prep(X_test_c22)

constant catch22 columns: 2 of 220


# Evaluation Metrics

In [8]:
def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def brier_skill_score(y_true, y_prob):
    base_rate = np.mean(y_true)
    brier_ref = brier_score_loss(y_true, np.full(len(y_true), base_rate))
    return 1 - brier_score_loss(y_true, y_prob) / brier_ref

def evaluate_model_metrics( 
        y_actual, 
        y_pred, 
        y_prob, 
        model_name="Model"
    ):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred, zero_division=0),
        "Recall": recall_score(y_actual, y_pred, zero_division=0),
        "F1-Score": f1_score(y_actual, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "Brier": brier_score_loss(y_actual, y_prob),
        "BSS" : brier_skill_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

def print_metrics(metrics):
    print("===================")
    print("PERFORMANCE METRICS")
    print("===================")
    for name, val in metrics.items():
        formatted_val = round(val,5) if isinstance(val, (float, np.floating)) else val   
        print(f"{name:<17}: {formatted_val}")
    print()

# Triplet Loss

### Loss Function

In [9]:
def triplet_loss(anchor, positive, negative, margin=1.0):
    d_ap = F.pairwise_distance(anchor, positive)   # distance from anchor to positive
    d_an = F.pairwise_distance(anchor, negative)   # distance from anchor to negative
    losses = F.relu(d_ap - d_an + margin)          # loss (0 is good. More positive the worse)
    return losses.mean()

### 1-D Convolutional Encoder

In [10]:
class CausalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.0):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation                
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel_size, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation)
        self.act = nn.LeakyReLU(0.01)      
        self.drop = nn.Dropout1d(dropout)                         
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()  # remap C_in -> C_out so shapes match

    def forward(self, x):                       # x                                       
        out = F.pad(x, (self.left_pad, 0))      # pad zeros on the left                   
        out = self.act(self.conv1(out))         # conv1 (C_in -> C_out), then LeakyReLU   
        out = F.pad(out, (self.left_pad, 0))    # pad zeros on the left                   
        out = self.act(self.conv2(out))         # conv2 (C_out -> C_out), then LeakyReLU  
        out = self.drop(out)                    # Dropout some of the neurons in training 
        return out + self.skip(x)               # changes + original                      

    
class TimeSeriesEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=32, depth=6, emb_dim=64, kernel_size=3, normalize=True, dropout=0.0):
        super().__init__()
        blocks = []
        for i in range(depth):
            blocks.append(
                CausalConvBlock(
                    in_ch if i == 0 else hidden_ch,  
                    hidden_ch,
                    kernel_size,
                    dilation=2 ** i,             
                    dropout=dropout,
                )
            )               
        self.blocks = nn.Sequential(*blocks)      
        self.linear = nn.Linear(2 * hidden_ch, emb_dim)
        self.normalize = normalize

    def forward(self, out):                     # x                             
        out = out.permute(0, 2, 1)              # swap channels and time        
        out = self.blocks(out)                  # lean C traces                 
        peaks = out.max(dim=2).values           # keep just peaks               
        latest = out[:, :, -1]                  # features at the final timestep
        out = torch.cat([peaks, latest], dim=1) # peaks + latest                
        out = self.linear(out)                  # remix to fingerprint          
        if self.normalize:
            out = F.normalize(out, dim=1)       # each row gets length 1        
        return out

### GRU Encoder

In [11]:
class GRUEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=32, depth=2, emb_dim=64, normalize=True, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(in_ch, hidden_ch, num_layers=depth, batch_first=True, dropout=dropout if depth > 1 else 0.0)
        self.linear = nn.Linear(hidden_ch, emb_dim)
        self.normalize = normalize

    def forward(self, x):        
        _, h = self.gru(x)       
        out = self.linear(h[-1]) 
        return F.normalize(out, dim=1) if self.normalize else out

### MLP Encoder

In [12]:
class MLPEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=64, depth=2, emb_dim=32, normalize=True, dropout=0.0):
        super().__init__()
        layers, d = [], in_ch
        for _ in range(depth):
            layers += [
                nn.Linear(d, hidden_ch), 
                nn.LeakyReLU(0.01), 
                nn.Dropout(dropout)
            ]
            d = hidden_ch
        self.net = nn.Sequential(*layers)
        self.linear = nn.Linear(d, emb_dim)
        self.normalize = normalize

    def forward(self, x):
        out = self.linear(self.net(x))
        return F.normalize(out, dim=1) if self.normalize else out

### Embedding Extraction

In [13]:
@torch.no_grad()
def embed(enc, X, batch_size=512):
    enc.eval()
    # Run through in sizes of batch_size at a time
    chunks = [enc(X[i:i + batch_size]) for i in range(0, len(X), batch_size)]
    return torch.cat(chunks).cpu().numpy()

### Triplet Mining

In [14]:
@torch.no_grad()
def refresh_hard_pool(enc, X_train_t, y, pool):
    y = np.asarray(y)
    sep_idx, nsep_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    enc.eval()
    Z = torch.cat([enc(X_train_t[i:i + 512]) for i in range(0, len(X_train_t), 512)])
    enc.train()
    sep_t = torch.as_tensor(sep_idx, device=Z.device)
    nsep_t = torch.as_tensor(nsep_idx, device=Z.device)
    d_to_sep = torch.cdist(Z[nsep_t], Z[sep_t]).min(dim=1).values  
    return nsep_idx[d_to_sep.argsort()[:pool].cpu().numpy()]

def sample_batch_indices(y, batch_size, rng,
                         sep_fraction=0.4,      # share of the batch that is SEP anchors
                         hard_frac=0.0,         # share of the NSEP slots filled from hard_pool
                         hard_pool=None,        # NSEP row numbers closest to the SEPs (from refresh_hard_pool)
                        ):
    y = np.asarray(y)

    # Separate dataset indices into SEP and NSEP pools
    sep_idx, nsep_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)

    # Compute sample size for batches
    n_sep = int(round(batch_size * sep_fraction))
    n_nsep = batch_size - n_sep
    n_hard = int(round(n_nsep * hard_frac)) if hard_pool is not None else 0

    # Sample SEP indices
    sep = rng.choice(sep_idx, n_sep, replace=n_sep > len(sep_idx))

    # Sample hard NSEP indices from the hard pool (if active)
    hard_nsep = (
        rng.choice(hard_pool, n_hard, replace=n_hard > len(hard_pool))
        if n_hard
        else np.empty(0, dtype=int)
    )

    # Sample the remaining majority slots from standard NSEPs (excluding hard ones to avoid duplicates)
    rest_pool = np.setdiff1d(nsep_idx, hard_nsep)  
    rest = rng.choice(
        rest_pool,
        n_nsep - n_hard,
        replace=(n_nsep - n_hard) > len(rest_pool),
    )

    # Combine all components
    return np.concatenate([sep, hard_nsep, rest])

@torch.no_grad()
def mine_semi_hard(emb, y, margin=0.3, include_hard=False, max_triplets=15000):
    # Compute pairwise Euclidean distance matrix for all embeddings in the batch
    d = torch.cdist(emb, emb)

    # Make a 3D pairwise distance tensor
    d_ap, d_an = d.unsqueeze(2), d.unsqueeze(1)

    # Create boolean masks to filter valid triplet combinations
    same = y[:, None] == y[None, :]  # True where labels match
    eye = torch.eye(len(y), dtype=torch.bool, device=y.device) 
    valid = (same & ~eye).unsqueeze(2) & (~same).unsqueeze(1)

    # Apply triplet mining boundary conditions based on margin
    # Semi-hard condition: negative is further than positive, but within the margin window
    mask = valid & (d_an < d_ap + margin)
    if not include_hard:
        mask &= (d_an > d_ap)

    # Extract indices of valid triplets
    a, p, n = mask.nonzero(as_tuple=True)

    # Check if the mined triplet count exceeds the safety cap
    if len(a) > max_triplets:
        keep = torch.randperm(len(a), device=a.device)[:max_triplets]
        a, p, n = a[keep], p[keep], n[keep]

    return a, p, n

### Classifier Heads

In [15]:
def pairwise_dist(A, B):
    # Euclidean distance between every row of A (n, d) and every row of B (m, d) -> (n, m).
    a2 = (A ** 2).sum(axis=1)[:, None]
    b2 = (B ** 2).sum(axis=1)[None, :]
    d2 = np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0)
    return np.sqrt(d2)

In [16]:
class Model:
    def fit(self, Z, y):
        self._fit(Z, y)
        raw = self._raw_train(Z)                       
        self._mu, self._sd = raw.mean(), raw.std() + 1e-12
        return self
    def score(self, Z): 
        return expit((self._raw(Z) - self._mu) / self._sd)
    def score_train(self, Z): 
        return expit((self._raw_train(Z) - self._mu) / self._sd)
    def _raw_train(self, Z):  
        return self._raw(Z)      

class LogRegModel(Model):
    def __init__(self, C=1.0, class_weight="balanced"):
        lr = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000)
        self.model = lr
    def _fit(self, Z, y):     
        self.model.fit(Z, y)
    def _raw(self, Z):        
        return self.model.decision_function(Z)
    def score(self, Z):       
        return self.model.predict_proba(Z)[:, 1]   
    def score_train(self, Z): 
        return self.score(Z)

class CentroidModel(Model):
    def __init__(self, use_nsep=True):
        self.use_nsep = use_nsep
    def _fit(self, Z, y):
        self.mu_sep  = Z[y == 1].mean(axis=0)
        self.mu_nsep = Z[y == 0].mean(axis=0)
    def _raw(self, Z):
        d_sep = np.linalg.norm(Z - self.mu_sep, axis=1)
        if not self.use_nsep:
            return -d_sep
        return np.linalg.norm(Z - self.mu_nsep, axis=1) - d_sep

class KnnModel(Model):
    def __init__(self, k=5):
        self.k = k
    def _fit(self, Z, y):
        self.sep_pos = np.flatnonzero(y == 1)
        self.Z_sep = Z[self.sep_pos]
    def _raw(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)
    def _raw_train(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        d[self.sep_pos, np.arange(len(self.sep_pos))] = np.inf 
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)

class SvmModel(Model):
    def __init__(self, kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"):
        self.model = SVC(kernel=kernel, C=C, gamma=gamma, class_weight=class_weight, cache_size=2000)
    def _fit(self, Z, y):  
        self.model.fit(Z, y)
    def _raw(self, Z):     
        return self.model.decision_function(Z)

class XgbModel(Model):
    def __init__(self, scale_pos_weight="balanced", max_depth=3, min_child_weight=5, n_estimators=200, learning_rate=0.05, subsample=0.7, colsample_bytree=0.5, reg_lambda=10, random_state=RAND_STATE):
        self.scale_pos_weight = scale_pos_weight
        self.params = dict(
            max_depth=max_depth, 
            min_child_weight=min_child_weight,
            n_estimators=n_estimators, 
            learning_rate=learning_rate,
            subsample=subsample, 
            colsample_bytree=colsample_bytree,
            reg_lambda=reg_lambda, 
            random_state=random_state, 
            n_jobs=-1
        )
    def _fit(self, Z, y):
        y = np.asarray(y)
        spw = (y == 0).sum() / max((y == 1).sum(), 1) if self.scale_pos_weight == "balanced" else self.scale_pos_weight
        self.model = XGBClassifier(scale_pos_weight=spw, **self.params)
        self.model.fit(Z, y)
    def _raw(self, Z):
        return self.model.predict(Z, output_margin=True)
    def score(self, Z):
        return self.model.predict_proba(Z)[:, 1]
    def score_train(self, Z):
        return self.score(Z)

HEADS = {"logreg": LogRegModel, "centroid": CentroidModel, "knn": KnnModel, "svm": SvmModel, "xgb": XgbModel}

def make_model(name, params=None):
    return HEADS[name](**(params or {}))

### Classifier Evaluation

In [17]:
def evaluate_classifier(head, head_params, Z_train, y_train, Z_val, y_val, Z_eval=None, y_eval=None, model_name="Model"):
    # Instantiate the model using factory function and fit on training embeddings
    clf = make_model(head, head_params).fit(Z_train, y_train)

    # Generate score predictions for training and validation sets
    s_train, s_val = clf.score_train(Z_train), clf.score(Z_val)

    # Find best threshold, maximize TSS
    fpr, tpr, thr = roc_curve(y_val, s_val)
    threshold = float(thr[np.argmax(tpr - fpr)])

    # Select the evaluation target (defaults to validation set if no test set is provided)
    if Z_eval is None:
        y_eval, s_eval = y_val, s_val
    else:
        s_eval = clf.score(Z_eval)

    # Evaluate
    y_pred = (s_eval >= threshold).astype(int)
    metrics = evaluate_model_metrics(y_eval, y_pred, s_eval, model_name=model_name)

    # Compute extra training metrics
    fpr_t, tpr_t, _ = roc_curve(y_train, s_train)
    metrics["train_ROC_AUC"] = roc_auc_score(y_train, s_train)
    metrics["train_TSS"] = float(np.max(tpr_t - fpr_t))
    metrics["Threshold"] = threshold
    return metrics

### Encoder Training

In [18]:
def evaluate_step(step, train_loss, enc, X_train_t, y_train, X_val_t, y_val, head, head_params):
    Z_train, Z_val = embed(enc, X_train_t), embed(enc, X_val_t)
    results = evaluate_classifier(head, head_params, Z_train, y_train, Z_val, y_val)
    enc.train()
    return {"step": step, "train_loss": train_loss, "TSS": results["TSS"], "train_TSS": results["train_TSS"], "ROC_AUC": results["ROC-AUC"], "train_ROC_AUC": results["train_ROC_AUC"]}

def print_step_metrics(metrics):
    print(f"step {metrics['step']:5d} | loss {metrics['train_loss']:.4f} | Train TSS {metrics['train_TSS']:.3f} | Val TSS {metrics['TSS']:.3f} | Train ROC-AUC {metrics['train_ROC_AUC']:.3f} | Val ROC-AUC {metrics['ROC_AUC']:.3f}")

In [19]:
def train_encoder(X_train_t, y_train, X_val_t, y_val, 
                  seed=RAND_STATE,          # Random state seed
                  n_steps=1000,             # Number of training steps
                  lr=1e-3,                  # Learning rate
                  batch_size=256,           # Number of anchors per step (triplets are mined inside the batch)
                  hidden_ch=32,             # Number of pattern detectors
                  emb_dim=32,               # Embedding dimension
                  weight_decay=0.01,        # L2 penalty: shrinks weights to 0
                  depth=6,                  # Number of stacked blocks
                  sep_fraction=0.5,         # Portion of each batch that is SEP
                  hard_frac=0.5,            # Portion of the NSEP slots drawn from the hard pool (0 = fully random batches)
                  hard_pool=2000,           # Hard pool = this many NSEPs closest to any SEP in embedding space
                  refresh_every=250,        # Recompute the hard pool from the current encoder every this many steps
                  include_hard=False,       # Also use negatives closer than the positive (not only semi-hard)
                  max_triplets=15000,       # Cap on mined triplets per step
                  margin=0.3,               # How much farther negative must be
                  eval_every=100,           # How often to evaluate 
                  dropout=0.0,              # Training dropout
                  head="logreg",            # which classifier the printed curves use
                  head_params=None,         # classifier's settings
                  encoder="gru"             # encoder selection
                ):

    device = X_train_t.device
    torch.manual_seed(seed)                 # controls the encoder's starting weights
    rng = np.random.default_rng(seed)       # controls which windows are sampled
    y_np = np.asarray(y_train)
    y_t = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Initialize encoder and optimizer (encoder's weights)
    if (encoder == "cnn"):
        enc = TimeSeriesEncoder(in_ch=X_train_t.shape[2], hidden_ch=hidden_ch, depth=depth, emb_dim=emb_dim, dropout=dropout).to(device)
    elif (encoder == "gru"):
        enc = GRUEncoder(in_ch=X_train_t.shape[2], hidden_ch=hidden_ch, depth=depth, emb_dim=emb_dim, dropout=dropout).to(device)
    elif (encoder == "mlp"):
        enc = MLPEncoder(in_ch=X_train_t.shape[-1], hidden_ch=hidden_ch, depth=depth, emb_dim=emb_dim, dropout=dropout).to(device)
    else:
        raise ValueError(f"No encoder found matching '{encoder}'")
    optimizer = torch.optim.Adam(enc.parameters(), lr=lr, weight_decay=weight_decay)

    # Evaluate untrained model (step 0)
    history = [evaluate_step(0, float("nan"), enc, X_train_t, y_train, X_val_t, y_val, head, head_params)]
    print_step_metrics(history[0])

    pool_idx = None
    recent_losses = []                           
    for step in range(1, n_steps + 1):
        # Periodically rebuild the pool of NSEPs that currently look most like SEPs
        if hard_frac > 0 and (step - 1) % refresh_every == 0:
            pool_idx = refresh_hard_pool(enc, X_train_t, y_np, hard_pool)

        # Pick a batch and embed it
        idx = torch.as_tensor(sample_batch_indices(y_np, batch_size, rng, sep_fraction, hard_frac, pool_idx), dtype=torch.long, device=device)
        emb = enc(X_train_t[idx])

        # Mine triplets from the batch embeddings, then apply the same triplet loss as before
        a, p, n = mine_semi_hard(emb.detach(), y_t[idx], margin=margin, include_hard=include_hard, max_triplets=max_triplets)
        loss = triplet_loss(emb[a], emb[p], emb[n], margin=margin) if len(a) else emb.sum() * 0

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        recent_losses.append(loss.item())

        # Evaluate on validation every `eval_every` steps, and at the very end
        if step % eval_every == 0 or step == n_steps:
            history.append(evaluate_step(step, float(np.mean(recent_losses)), enc, X_train_t, y_train, X_val_t, y_val, head, head_params))
            recent_losses = []
            print_step_metrics(history[-1])

    return enc, history

## Experiments

### Experiment Runner

In [20]:
RESULTS = {} # Stores all run results for later comparison

In [21]:
def run_triplet(cfg, seed, X_eval_t, y_eval):
    # Pick inputs: the MLP reads catch22 features, the CNN/GRU read the raw normalized windows
    if cfg["encoder"] == "mlp":
        Xtr, Xva = C22_train_t, C22_val_t
        if X_eval_t is X_val_t:
            Xev = C22_val_t
        elif X_eval_t is X_test_t:
            Xev = C22_test_t
        else:
            raise ValueError("X_eval_t must be X_val_t or X_test_t")
    else:
        Xtr, Xva, Xev = X_train_t, X_val_t, X_eval_t
    
    # Train encoder on train, fit head on train, pick threshold on val, score on the eval set.
    enc, _ = train_encoder(Xtr, y_train, Xva, y_val, seed=seed, **cfg)

    Z_train = embed(enc, Xtr)
    Z_val = embed(enc, Xva)
    Z_eval = embed(enc, Xev)
    return evaluate_classifier(cfg["head"], cfg["head_params"], Z_train, y_train, Z_val, y_val, Z_eval, y_eval, model_name=f"Triplet+{cfg['head']} seed {seed}")

def run_experiment(name, grid, n_search_seeds, n_final_runs, select_metric):
    print("═" * 70)
    print(f"{name}: searching {len(grid)} configs x {n_search_seeds} seeds on validation")
    print("═" * 70)

    # Loop through configs and seeds. Train and eval
    rows = []
    for c, cfg in enumerate(grid):
        for i in range(n_search_seeds):
            seed = RAND_STATE + i
            # Print run header
            header_label = f"\n[config {c+1}/{len(grid)} | seed {seed}] {cfg}"
            print(header_label)

            # Run training and eval
            t0 = time.perf_counter()      
            m = run_triplet(cfg, seed, X_val_t, y_val)
            secs = time.perf_counter() - t0 

            # Add to df and print results
            row_data = {"config": c, "seed": seed}
            row_data.update(m)
            rows.append(row_data)
            print(f"Results: Val TSS={m['TSS']:.4f} | ROC-AUC={m['ROC-AUC']:.4f} | Thr={m['Threshold']:.4f} | {secs:.0f}s")

    search_df = pd.DataFrame(rows)
    summary = search_df.groupby("config")[["TSS", "ROC-AUC"]].agg(["mean", "std"]).round(3)
    print(summary.rename(index=lambda c: c + 1))

    best_c = int(search_df.groupby("config")[select_metric].mean().idxmax())
    best_cfg = grid[best_c]
    print(f"\nBest by val {select_metric}: config #{best_c+1} | {best_cfg}")

    # Run on test set
    print(f"\nRunning {n_final_runs} final runs on test")
    print("─" * 70)
    final_rows = []
    for run in range(n_final_runs):
        seed = RAND_STATE + 100 + run
        print(f"\nRun {run+1} (seed {seed})", flush=True)

        t0 = time.perf_counter()
        m = run_triplet(best_cfg, seed, X_test_t, y_test)
        secs = time.perf_counter() - t0 

        row_data = {"config": best_c, "seed": seed}
        row_data.update(m)
        final_rows.append(row_data)
        print_metrics(m)

    final_df = pd.DataFrame(final_rows)
    print(f"\n{name} test: TSS {final_df['TSS'].mean():.3f} ± {final_df['TSS'].std():.3f} | AUC {final_df['ROC-AUC'].mean():.3f} ± {final_df['ROC-AUC'].std():.3f}")
    return {"name": name, "best_cfg": best_cfg, "search": search_df, "final": final_df}

### Config

In [22]:
BASE_CONFIG = dict(
    head='logreg', 
    encoder='gru',

    # loss / regularization
    margin=0.443, 
    weight_decay=0.0136,
    dropout=0.0,

    # encoder
    hidden_ch=64, 
    emb_dim=32, 
    depth=2,

    # triplets
    batch_size=256,
    sep_fraction=0.5, 
    hard_frac=0.66,
    hard_pool=2000,
    refresh_every=250,
    include_hard=True,
    max_triplets=15000,

    # training
    lr=1.1e-05, 
    n_steps=1000, 
    eval_every=1000,

    # head specific parameters
    head_params={'C': 0.01},
)

### Logistic Regression

In [23]:
LOGREG_BASE = {**BASE_CONFIG, "head": "logreg"}

rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**LOGREG_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-2.5, -0.9), 5)
    cfg["margin"] = round(rng.uniform(0.2, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.2, 0.3, 0.4, 0.5]))
    cfg["hidden_ch"] = int(rng.choice([32, 64, 128]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))

    RANDOM_CFGS.append(cfg)

LOGREG_GRID = [
    *[{**c, "encoder": "cnn", "n_steps": 0} for c in RANDOM_CFGS],  # untrained CNN reference
    *[{**c, "encoder": "gru", "n_steps": 0} for c in RANDOM_CFGS],  # untrained GRU reference
    *[{**c, "encoder": "mlp", "n_steps": 0} for c in RANDOM_CFGS],  # untrained MLP reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],    # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on GRU
    *[{**c, "encoder": "mlp", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on MLP
    {**LOGREG_BASE}                                                 # base config on GRU
]

RESULTS["logreg"] = run_experiment(
    name="Logistic Regression", 
    grid=LOGREG_GRID,
    n_search_seeds=3,
    n_final_runs=3,
    select_metric="TSS"
)

══════════════════════════════════════════════════════════════════════
Logistic Regression: searching 91 configs x 3 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/91 | seed 42] {'head': 'logreg', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 'refresh_every': 250, 'include_hard': True, 'max_triplets': 15000, 'lr': 0.000202, 'n_steps': 0, 'eval_every': 1000, 'head_params': {'C': 0.01}}
step     0 | loss nan | Train TSS 0.516 | Val TSS 0.604 | Train ROC-AUC 0.804 | Val ROC-AUC 0.833
Results: Val TSS=0.6038 | ROC-AUC=0.8334 | Thr=0.5580 | 12s

[config 1/91 | seed 43] {'head': 'logreg', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 're

### Centroid Distance

In [24]:
CENTROID_BASE = {**BASE_CONFIG, "head": "centroid", "head_params": {}}

rng = np.random.default_rng(RAND_STATE + 1)

RANDOM_CFGS = []        

for i in range(15):
    cfg = {**CENTROID_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-2.5, -0.9), 5)
    cfg["margin"] = round(rng.uniform(0.2, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.2, 0.3, 0.4, 0.5]))
    cfg["hidden_ch"] = int(rng.choice([32, 64, 128]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))

    RANDOM_CFGS.append(cfg)

CENTROID_GRID = [
    *[{**c, "encoder": "cnn", "n_steps": 0} for c in RANDOM_CFGS],  # untrained CNN reference
    *[{**c, "encoder": "gru", "n_steps": 0} for c in RANDOM_CFGS],  # untrained GRU reference
    *[{**c, "encoder": "mlp", "n_steps": 0} for c in RANDOM_CFGS],  # untrained MLP reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],    # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on GRU
    *[{**c, "encoder": "mlp", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on MLP
    {**CENTROID_BASE}                                               # base config on GRU
]


RESULTS["centroid"] = run_experiment(
    name="Centroid",
    grid=CENTROID_GRID,
    n_search_seeds=3,
    n_final_runs=3,
    select_metric="TSS",
)

══════════════════════════════════════════════════════════════════════
Centroid: searching 91 configs x 3 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/91 | seed 42] {'head': 'centroid', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 'refresh_every': 250, 'include_hard': True, 'max_triplets': 15000, 'lr': 0.000202, 'n_steps': 0, 'eval_every': 1000, 'head_params': {}}
step     0 | loss nan | Train TSS 0.436 | Val TSS 0.546 | Train ROC-AUC 0.771 | Val ROC-AUC 0.819
Results: Val TSS=0.5463 | ROC-AUC=0.8186 | Thr=0.8087 | 2s

[config 1/91 | seed 43] {'head': 'centroid', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 'refresh_every': 250

### kNN

In [25]:
KNN_BASE = {**BASE_CONFIG, "head": "knn"}

rng = np.random.default_rng(RAND_STATE + 1)
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**KNN_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-2.5, -0.9), 5)
    cfg["margin"] = round(rng.uniform(0.2, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.2, 0.3, 0.4, 0.5]))
    cfg["hidden_ch"] = int(rng.choice([32, 64, 128]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))
    cfg["head_params"] = {"k": int(head_rng.choice([20, 40, 60]))}

    RANDOM_CFGS.append(cfg)

KNN_GRID = [
    *[{**c, "encoder": "cnn", "n_steps": 0} for c in RANDOM_CFGS],  # untrained CNN reference
    *[{**c, "encoder": "gru", "n_steps": 0} for c in RANDOM_CFGS],  # untrained GRU reference
    *[{**c, "encoder": "mlp", "n_steps": 0} for c in RANDOM_CFGS],  # untrained MLP reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],    # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on GRU
    *[{**c, "encoder": "mlp", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on MLP
    {**KNN_BASE}                                               # base config on GRU
]

RESULTS["knn"] = run_experiment(
    name="kNN",
    grid=KNN_GRID,
    n_search_seeds=3,
    n_final_runs=3,
    select_metric="TSS",
)

══════════════════════════════════════════════════════════════════════
kNN: searching 91 configs x 3 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/91 | seed 42] {'head': 'knn', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 'refresh_every': 250, 'include_hard': True, 'max_triplets': 15000, 'lr': 0.000202, 'n_steps': 0, 'eval_every': 1000, 'head_params': {'k': 40}}
step     0 | loss nan | Train TSS 0.195 | Val TSS 0.384 | Train ROC-AUC 0.602 | Val ROC-AUC 0.724
Results: Val TSS=0.3840 | ROC-AUC=0.7238 | Thr=0.7725 | 2s

[config 1/91 | seed 43] {'head': 'knn', 'encoder': 'cnn', 'margin': 0.208, 'weight_decay': 0.00372, 'dropout': 0.3, 'hidden_ch': 128, 'emb_dim': 16, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.18, 'hard_pool': 1592, 'refresh_every': 250, 'inclu

TypeError: KnnModel.__init__() got an unexpected keyword argument 'C'

### SVM

In [ ]:
SVM_BASE = {**BASE_CONFIG, "head": "svm", "head_params": {"kernel": "rbf"}}

rng = np.random.default_rng(RAND_STATE + 1)
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**SVM_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-2.5, -0.9), 5)
    cfg["margin"] = round(rng.uniform(0.2, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.2, 0.3, 0.4, 0.5]))
    cfg["hidden_ch"] = int(rng.choice([32, 64, 128]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))
    cfg["head_params"] = {
        "C": round(10 ** head_rng.uniform(-2, 1), 6),
        "gamma": round(10 ** head_rng.uniform(-4, -2), 6),
    }

    RANDOM_CFGS.append(cfg)

SVM_GRID = [
    *[{**c, "encoder": "cnn", "n_steps": 0} for c in RANDOM_CFGS],  # untrained CNN reference
    *[{**c, "encoder": "gru", "n_steps": 0} for c in RANDOM_CFGS],  # untrained GRU reference
    *[{**c, "encoder": "mlp", "n_steps": 0} for c in RANDOM_CFGS],  # untrained MLP reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],    # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on GRU
    *[{**c, "encoder": "mlp", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on MLP
    {**SVM_BASE}                                                    # base config on GRU
]

RESULTS["svm"] = run_experiment(
    name="SVM",
    grid=SVM_GRID,
    n_search_seeds=3,
    n_final_runs=3,
    select_metric="TSS",
)

### XGBoost

In [ ]:
XGB_BASE = {**BASE_CONFIG, "head": "xgb", "head_params": {"max_depth": 3, "min_child_weight": 5, "n_estimators": 200, "learning_rate": 0.05, "colsample_bytree": 0.5,}}

rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**XGB_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-2.5, -0.9), 5)
    cfg["margin"] = round(rng.uniform(0.2, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.2, 0.3, 0.4, 0.5]))
    cfg["hidden_ch"] = int(rng.choice([32, 64, 128]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))
    cfg["head_params"] = {
        "max_depth": int(rng.choice([2, 3, 4, 6])),
        "min_child_weight": int(rng.choice([10, 20, 40, 800])),
        "n_estimators": int(rng.choice([50, 100, 200])),
        "learning_rate": round(10 ** rng.uniform(-2.5, -1), 4),
        "colsample_bytree": float(rng.choice([0.3, 0.5])),
    }
    RANDOM_CFGS.append(cfg)

XGB_GRID = [
    *[{**c, "encoder": "cnn", "n_steps": 0} for c in RANDOM_CFGS],  # untrained CNN reference
    *[{**c, "encoder": "gru", "n_steps": 0} for c in RANDOM_CFGS],  # untrained GRU reference
    *[{**c, "encoder": "mlp", "n_steps": 0} for c in RANDOM_CFGS],  # untrained MLP reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],    # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on GRU
    *[{**c, "encoder": "mlp", "depth": 2} for c in RANDOM_CFGS],    # random cfgs on MLP
    {**XGB_BASE}                                                    # base config on GRU
]

RESULTS["xgb"] = run_experiment(
    name="XGBoost",
    grid=XGB_GRID,
    n_search_seeds=3,
    n_final_runs=3,
    select_metric="TSS",
)

In [ ]:
# Pick the best classifier by mean test TSS across its final runs
BEST_KEY = max(RESULTS, key=lambda k: RESULTS[k]["final"]["TSS"].mean())
BEST_CONFIG = RESULTS[BEST_KEY]["best_cfg"]
print(f"Best head: {BEST_KEY} | test TSS {RESULTS[BEST_KEY]['final']['TSS'].mean():.3f}")
print(BEST_CONFIG)

# Ablation Test

In [ ]:
# Full config chosen because it was the best of above random search
ABL_SEEDS = [RAND_STATE + 100 + i for i in range(8)]   

# Various Ablation Test Configs
COMPONENT_ABLATIONS = {        
    "full":              {},
    "untrained":         {"n_steps": 0},
    "no_hard_pool":      {"hard_frac": 0.0},
    "no_hard_negatives": {"include_hard": False},
    "no_dropout":        {"dropout": 0.0},
    "no_weight_decay":   {"weight_decay": 0.0},
}
HYPERPARAMETERS_ABLATIONS = {                  
    "margin=0.2":        {"margin": 0.2},
    "margin=0.6":        {"margin": 0.6},
    "emb_dim=16":        {"emb_dim": 16},
    "hidden_ch=32":      {"hidden_ch": 32},
    "hidden_ch=128":     {"hidden_ch": 128},
}
LR_ABLATIONS = {          
    "lr x10":            {"lr": 1.1e-04},
    "lr x100":           {"lr": 1.1e-03},
    "steps x3":          {"n_steps": 3000},
}

In [ ]:
def run_variants(variants, base_cfg=BEST_CONFIG, seeds=ABL_SEEDS, X_eval=None, y_eval=None):
    X_eval = X_val_t if X_eval is None else X_eval
    y_eval = y_val if y_eval is None else y_eval
    rows = []
    for name, overrides in variants.items():
        cfg = {**base_cfg, **overrides}
        for seed in seeds:
            t0 = time.perf_counter()
            with contextlib.redirect_stdout(io.StringIO()):
                m = run_triplet(cfg, seed, X_eval, y_eval)
            rows.append({"variant": name, "seed": seed, "TSS": m["TSS"], "ROC-AUC": m["ROC-AUC"],
                         "train_TSS": m["train_TSS"]})
            print(f"{name:<20} seed {seed} | TSS {m['TSS']:.3f} | AUC {m['ROC-AUC']:.3f} | {time.perf_counter()-t0:.0f}s", flush=True)
    return pd.DataFrame(rows)

def paired_summary(df, ref="full", metric="TSS"):
    wide = df.pivot(index="seed", columns="variant", values=metric)
    out = []
    for v in wide.columns:
        d = (wide[v] - wide[ref]).dropna()
        n = len(d)
        sem = d.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
        ci = stats.t.ppf(0.975, n - 1) * sem if n > 1 else np.nan
        out.append({"variant": v, f"{metric} mean": wide[v].mean(), f"{metric} std": wide[v].std(ddof=1),
                    "diff vs ref": d.mean(), "95% CI ±": ci,
                    "seeds better": int((d > 0).sum()), "n": n,
                    "distinguishable": bool(n > 1 and abs(d.mean()) > ci)})
    return pd.DataFrame(out).set_index("variant").round(3)

In [ ]:
# ==== LEAVE ONE OUT TESTS ====
# Component Ablation Test
df_comp = run_variants(COMPONENT_ABLATIONS)
print("\n=== COMPONENT ABLATION ===")
print(paired_summary(df_comp, metric="TSS"))
print(paired_summary(df_comp, metric="ROC-AUC"))

# Hyperparamerter Ablation Test
df_sens = run_variants({"full": {}, **HYPERPARAMETERS_ABLATIONS})
print("\n=== HYPERPARAMETER ABLATION ===")
print(paired_summary(df_sens, metric="TSS"))

# Learning Rate Ablation Test
df_diag = run_variants({"full": {}, "untrained": {"n_steps": 0}, **LR_ABLATIONS})
print("\n=== LEARNING RATE ===")
print(paired_summary(df_diag, metric="TSS"))

# === ADD BACK IN TESTS ===
STRIPPED = {"hard_frac": 0.0, "include_hard": False, "dropout": 0.0, "weight_decay": 0.0}
ADD_BACK = {
    "stripped":         {},
    "+hard_pool":       {"hard_frac": BEST_CONFIG["hard_frac"]},
    "+hard_negatives":  {"include_hard": True},
    "+dropout":         {"dropout": BEST_CONFIG["dropout"]},
    "+weight_decay":    {"weight_decay": BEST_CONFIG["weight_decay"]},
    "all":      {k: BEST_CONFIG[k] for k in STRIPPED},
}
df_add = run_variants(ADD_BACK, base_cfg={**BEST_CONFIG, **STRIPPED})
print("\n=== ADD-BACK ===")
print(paired_summary(df_add, ref="stripped", metric="TSS"))

In [ ]:
# Final test on test set for final report
TEST_CFGS = {   
    "full": {}, 
    "untrained": {"n_steps": 0},
    "no_hard_pool": {"hard_frac": 0.0},
    "no_dropout":   {"dropout": 0.0},
}
TEST_SEEDS = [RAND_STATE + 300 + i for i in range(5)]
df_test = run_variants(TEST_CFGS, seeds=TEST_SEEDS, X_eval=X_test_t, y_eval=y_test)
print(paired_summary(df_test, metric="TSS"))

# t-SNE Visualization

In [ ]:
def train_and_embed(cfg, seed, n_steps=None, label=""):
    cfg = {**cfg, "n_steps": n_steps}         
    print(f"\n=== {label} (n_steps={cfg['n_steps']}) ===")

    # MLP reads catch22 features, CNN/GRU read raw windows
    if cfg["encoder"] == "mlp":
        Xtr, Xva, Xte = C22_train_t, C22_val_t, C22_test_t
    else:
        Xtr, Xva, Xte = X_train_t, X_val_t, X_test_t

    enc, _ = train_encoder(Xtr, y_train, Xva, y_val, seed=seed, **cfg)
    Z = {"train": embed(enc, Xtr), "val": embed(enc, Xva), "test": embed(enc, Xte)}
    metrics = evaluate_classifier(cfg["head"], cfg["head_params"], Z["train"], y_train, Z["val"], y_val, Z["test"], y_test, model_name=label)
    return Z, metrics

# Get embeddings and metrics
Z_before, m_before = train_and_embed(BEST_CONFIG, RAND_STATE, n_steps=0, label="Untrained")
Z_after, m_after = train_and_embed(BEST_CONFIG, RAND_STATE, n_steps=BEST_CONFIG["n_steps"], label="Trained")

print(f"\nUntrained TSS={m_before['TSS']:.3f}  ROC-AUC={m_before['ROC-AUC']:.3f}")
print(f"Trained TSS={m_after['TSS']:.3f}  ROC-AUC={m_after['ROC-AUC']:.3f}")

In [ ]:
def run_tsne(Z, perplexity=30, seed=RAND_STATE):
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        metric="euclidean", 
        random_state=seed,
    ).fit_transform(Z)

tsne_before = run_tsne(Z_before["test"])
tsne_after  = run_tsne(Z_after["test"])

In [ ]:
def scatter_tsne(ax, emb2d, y, title):
    nsep = y == 0
    sep = y == 1

    ax.scatter(emb2d[nsep, 0], emb2d[nsep, 1], s=20, c="#2a80db", alpha=0.35, linewidths=0, label=f"NSEP (n={nsep.sum()})", rasterized=True)
    ax.scatter(emb2d[sep, 0], emb2d[sep, 1], s=55, c="#d62728", edgecolors="black", linewidths=0.6, label=f"SEP (n={sep.sum()})", zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", frameon=True, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
scatter_tsne(axes[0], tsne_before, y_test, "Untrained")
scatter_tsne(axes[1], tsne_after, y_test, "Trained")
plt.tight_layout()
plt.show()